# 03: Generate GTFS from ML Model

Load the trained XGBoost model (with `direction` feature) from Google Drive, generate all GTFS files using ParkGo source data, and export macau-gtfs.zip.

In [ ]:
!pip install pandas numpy requests pyproj joblib xgboost haversine

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import joblib, os, json, zipfile
import pandas as pd
import numpy as np
import requests
from pyproj import Transformer
from haversine import haversine, Unit

model_dir = "/content/drive/MyDrive/parkgo_gtfs_model"
model = joblib.load(os.path.join(model_dir, "bus_trip_time_model.pkl"))
encoders = {
    "route": joblib.load(os.path.join(model_dir, "encoder_route.pkl")),
    "zone_type": joblib.load(os.path.join(model_dir, "encoder_zone_type.pkl")),
}
feature_cols = joblib.load(os.path.join(model_dir, "feature_cols.pkl"))
print("Model and encoders loaded")
print("Features:", feature_cols)

In [ ]:
BASE = "https://raw.githubusercontent.com/ChiHin-Lio/macau-gtfs-data/main"

# 1. bus-stops.json
stops = requests.get(f"{BASE}/bus-stops.json").json()
stop_dict = {}
for s in stops:
    stop_dict[s["id"]] = s
# Add missing stops referenced by route-stops
for sid, lat, lon in [("M280", 22.2025, 113.5461), ("M9/3", 22.1944, 113.5392), ("T345/9", 22.1600, 113.5600)]:
    if sid not in stop_dict:
        stop_dict[sid] = {"id": sid, "coordinates": [lon, lat], "nameCn": sid}
        stops.append({"id": sid, "coordinates": [lon, lat], "nameCn": sid})
print(f"Stops (patched): {len(stop_dict)}")

# 2. route-metadata.json
route_meta = requests.get(f"{BASE}/route-metadata.json").json()
print(f"Routes: {len(route_meta['routes'])}")

# 3. operators.json
operators = requests.get(f"{BASE}/operators.json").json()
print(f"Operators: {len(set(operators.values()))}")

# 4. route-stops.json
route_stops_data = requests.get(f"{BASE}/route-stops.json").json()
print(f"Route-stop mappings: {len(route_stops_data)}")

# 5. (Optional) DSAT official BUS_POLE for WGS84 coords
try:
    dsat_stops = pd.read_csv(f"{BASE}/dsat-data/BUS_POLE.csv")
    dsat_coords = {}
    for _, row in dsat_stops.iterrows():
        alias = str(row["P_ALIAS"]).strip()
        if pd.notna(row["lng"]) and pd.notna(row["lat"]):
            dsat_coords[alias] = (row["lat"], row["lng"])
    print(f"DSAT coords loaded: {len(dsat_coords)}")
except Exception as e:
    print(f"DSAT coords not available: {e}")
    dsat_coords = {}

In [ ]:
# agency.txt
agency_df = pd.DataFrame([
    {"agency_id": "TCM", "agency_name": "澳巴",
     "agency_url": "https://tcm.com.mo", "agency_timezone": "Asia/Macau"},
    {"agency_id": "TRANSMAC", "agency_name": "新福利",
     "agency_url": "https://transmac.com.mo", "agency_timezone": "Asia/Macau"},
])
agency_df.to_csv("agency.txt", index=False)
print("agency.txt written")

In [ ]:
# stops.txt — with WGS84 coords (already WGS84 in bus-stops.json)
stops_rows = []
for s in stops:
    sid = s["id"]
    x, y = s["coordinates"]
    
    # Prefer DSAT official coordinates if available
    if sid in dsat_coords:
        lat, lon = dsat_coords[sid]
    elif x != 0 and y != 0:
        lon, lat = x, y  # already WGS84 (lon, lat)
    else:
        continue
    
    stops_rows.append({
        "stop_id": sid,
        "stop_name": s.get("nameCn", sid),
        "stop_lat": round(lat, 6),
        "stop_lon": round(lon, 6),
    })

stops_df = pd.DataFrame(stops_rows)
stops_df.to_csv("stops.txt", index=False)
print(f"stops.txt written: {len(stops_df)} stops")

In [ ]:
# routes.txt — using ParkGo Chinese names
agency_map = {"澳巴": "TCM", "新福利": "TRANSMAC"}
route_ids = sorted(route_meta["routes"].keys(), key=lambda x: (len(x), x))

routes_rows = []
for rid in route_ids:
    meta = route_meta["routes"][rid]
    op_name = operators.get(rid, "澳巴")
    routes_rows.append({
        "route_id": rid,
        "agency_id": agency_map.get(op_name, "TCM"),
        "route_short_name": rid,
        "route_long_name": meta["nameCht"],
        "route_type": 3,
    })

routes_df = pd.DataFrame(routes_rows)
routes_df.to_csv("routes.txt", index=False)
print(f"routes.txt written: {len(routes_df)} routes")

In [ ]:
# calendar.txt — 12-month validity (internal use)
calendar_df = pd.DataFrame([
    {"service_id": "weekday",
     "monday":1,"tuesday":1,"wednesday":1,"thursday":1,"friday":1,
     "saturday":0,"sunday":0,
     "start_date":"20260701", "end_date":"20270630"},
    {"service_id": "weekend",
     "monday":0,"tuesday":0,"wednesday":0,"thursday":0,"friday":0,
     "saturday":1,"sunday":1,
     "start_date":"20260701", "end_date":"20270630"},
])
calendar_df.to_csv("calendar.txt", index=False)
print("calendar.txt written")

In [ ]:
# trips.txt — generate departure times from schedule
def parse_freq(freq_str):
    parts = str(freq_str).split("-")
    return sum(int(p) for p in parts) / len(parts)

def gen_times(first, last, periods, service_day):
    """Generate departure times using ALL periods for the given service_day."""
    matching = [p for p in periods if p.get("d","").strip() == service_day]
    if not matching:
        matching = periods
    if not matching:
        sh,sm = map(int, first.split(":")); eh,em = map(int, last.split(":"))
        s = sh+sm/60; e = eh+em/60
        if e <= s: e += 24
        t = s; times = []
        while t <= e:
            h,m = int(t)%24, int((t-int(t))*60)
            times.append(f"{h:02d}:{m:02d}"); t += 10/60
        return times
    times = []
    for p in matching:
        sh,sm = map(int, p["s"].split(":")); eh,em = map(int, p["e"].split(":"))
        s = sh+sm/60; e = eh+em/60
        if e <= s: e += 24
        f_str = str(p.get("f","")).strip()
        if not f_str or not f_str.replace("-","").replace(".","").isdigit():
            f_avg = 10
        else:
            ps = f_str.split("-")
            f_avg = sum(int(x) for x in ps) / len(ps)
        t = s
        while t < e:
            h,m = int(t)%24, int((t-int(t))*60)
            times.append(f"{h:02d}:{m:02d}"); t += f_avg/60
    return sorted(set(times))

trip_id_counter = 0
trips_rows = []

for rid in route_ids:
    meta = route_meta["routes"].get(rid, {})
    schedule = meta.get("schedule", {})
    for dir_id, dir_info in schedule.items():
        first = dir_info.get("first", "06:00")
        last = dir_info.get("last", "23:00")
        periods = dir_info.get("periods", [])
        dep_times = gen_times(first, last, periods, "weekdays")
        for dep in dep_times:
            trip_id_counter += 1
            trip_id = f"{rid}_{dir_id}_{trip_id_counter:04d}"
            trips_rows.append({
                "route_id": rid,
                "service_id": "weekday",
                "trip_id": trip_id,
                "direction_id": dir_id,
                "departure": dep,
            })

trips_df = pd.DataFrame(trips_rows)
trips_df.to_csv("trips.txt", index=False, columns=["route_id","service_id","trip_id","direction_id"])
print(f"trips.txt written: {len(trips_df)} trips")

In [ ]:
# stop_times.txt — use ML model to predict per-segment trip times
# v2: pre-compute segment times + safe_encode for unseen routes

def get_zone(stop_id):
    prefix = str(stop_id).split("/")[0]
    if prefix in ["M1","M2","M3"]: return "old_town"
    elif prefix in ["M4","M6"]: return "bridge"
    elif prefix in ["M7","M8"]: return "cotai"
    elif str(stop_id).startswith("T"): return "taipa"
    return "other"

def safe_encode(encoder, val):
    try: return encoder.transform([val])[0]
    except: return 0  # unseen route fallback

stop_times_rows = []
MAX_TRIP_TIME_S = 7200

# Pre-compute segment times per route for performance
route_segments = {}  # route_id -> list of segment travel times
for rid in route_ids:
    stop_seq = route_stops_data.get(rid, [])
    if len(stop_seq) < 2: continue
    segs = []
    for i in range(len(stop_seq) - 1):
        a, b = stop_seq[i], stop_seq[i+1]
        sa, sb = stop_dict.get(a, {}), stop_dict.get(b, {})
        ca, cb = sa.get("coordinates"), sb.get("coordinates")
        if ca and cb and ca[0] != 0 and cb[0] != 0:
            dist = haversine((ca[1], ca[0]), (cb[1], cb[0]), unit=Unit.KILOMETERS)
            try:
                feat = pd.DataFrame([[
                    dist, 12, 0, 0, 0,
                    safe_encode(encoders["route"], rid),
                    safe_encode(encoders["zone_type"], get_zone(a)),
                ]], columns=feature_cols)
                segs.append(max(model.predict(feat)[0], 15))
            except:
                segs.append(max(dist * 60 + 30, 15))
        else:
            segs.append(30)
    route_segments[rid] = segs

for _, trip in trips_df.iterrows():
    rid = trip["route_id"]
    dir_id = int(trip["direction_id"])
    trip_id = trip["trip_id"]
    dep_str = trip["departure"]
    
    stop_seq = route_stops_data.get(rid, [])
    segs = route_segments.get(rid, [])
    if len(stop_seq) < 2: continue
    
    parts = dep_str.split(":")
    h, m = int(parts[0]), int(parts[1])
    current_seconds = h * 3600 + m * 60
    cumulative = 0
    
    for seq_idx in range(len(stop_seq)):
        stop_id = stop_seq[seq_idx]
        if cumulative >= MAX_TRIP_TIME_S: break
        arrival_s = current_seconds + cumulative
        hh = int(arrival_s // 3600) % 24
        mm = int(arrival_s % 3600 // 60)
        ss = int(arrival_s % 60)
        time_str = f"{hh:02d}:{mm:02d}:{ss:02d}"
        
        stop_times_rows.append({
            "trip_id": trip_id,
            "arrival_time": time_str,
            "departure_time": time_str,
            "stop_id": stop_id,
            "stop_sequence": seq_idx,
        })
        
        if seq_idx < len(stop_seq) - 1:
            cumulative += segs[seq_idx] if seq_idx < len(segs) else 30

ST_df = pd.DataFrame(stop_times_rows)
ST_df.to_csv("stop_times.txt", index=False)
print(f"stop_times.txt written: {len(ST_df)} records")

In [ ]:
# Zip all GTFS files
gtfs_files = ["agency.txt", "stops.txt", "routes.txt", "trips.txt",
              "stop_times.txt", "calendar.txt"]
output_zip = "macau-gtfs.zip"

with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in gtfs_files:
        if os.path.exists(fn):
            zf.write(fn)

print(f"Created {output_zip}")
print(f"Size: {os.path.getsize(output_zip) / 1024:.1f} KB")

from google.colab import files
files.download(output_zip)